In [8]:
from Value import Value
from NN import NN
from graph import draw_graph

In [ ]:
# MNIST classifier
from tensorflow.keras.datasets import mnist

# Y is the labels, x is the images
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Number of images to train on
n_train = 1000
# Number of training loops
epochs = 100

x_train = x_train[:n_train]
y_train = y_train[:n_train]
dataset = x_train.reshape(n_train, -1) / 255.0
# Convert to Value objects
dataset = [[Value(pixel) for pixel in image] for image in dataset]
labels = y_train

# 3 layers with 16 neurons each
mlp = NN(784, [16, 16, 16, 10])
# Learning rate for SGD
lr = .01

In [11]:
# Training loop
for i in range(epochs):
    avg_loss = 0

    for j in range(n_train):
        # Zero gradients
        for param in mlp.get_params():
            param.grad = 0

        # Forward pass
        logits = mlp(dataset[j])

        # Normalize logits to probabilities
        denom = sum(logit.exp() for logit in logits)

        probs = [logit.exp() / denom for logit in logits]

        for k, prob in enumerate(probs):
            prob._op = "softmax"
            prob.label = f"prob_{k}"

        # Get cross-entropy loss
        loss = probs[labels[j]].log() * -1
        avg_loss += loss.v

        # Backward pass
        loss.backward()
        mlp.grad_descent(lr)
    
    avg_loss = avg_loss / n_train
    print(f"Epoch {i + 1},   Loss: {avg_loss:.6f}")

Epoch 1,   Loss: 1.676050
Epoch 2,   Loss: 1.642359
Epoch 3,   Loss: 1.638675
Epoch 4,   Loss: 1.618304
Epoch 5,   Loss: 1.635150
Epoch 6,   Loss: 1.627315
Epoch 7,   Loss: 1.596422
Epoch 8,   Loss: 1.544098
Epoch 9,   Loss: 1.560626
Epoch 10,   Loss: 1.529207
Epoch 11,   Loss: 1.519201
Epoch 12,   Loss: 1.539424
Epoch 13,   Loss: 1.532008
Epoch 14,   Loss: 1.520113
Epoch 15,   Loss: 1.525363
Epoch 16,   Loss: 1.512616
Epoch 17,   Loss: 1.513519
Epoch 18,   Loss: 1.518590
Epoch 19,   Loss: 1.509404
Epoch 20,   Loss: 1.509960
Epoch 21,   Loss: 1.515391
Epoch 22,   Loss: 1.500524
Epoch 23,   Loss: 1.525810


KeyboardInterrupt: 

In [12]:
# Testing the model
import numpy as np

# Number of images to test on
n_test = 2000

x_test = x_test[:n_test]
y_test = y_test[:n_test]
testset = x_test.reshape(n_test, -1) / 255.0
# Convert to Value objects
testset = [[Value(pixel) for pixel in image] for image in testset]
labels = y_test

n_correct = 0

for i in range(n_test):
    # Forward pass
    logits = mlp(testset[i])
    # Use scalar values instead of Values objects for speed
    logit_values = np.array([l.v for l in logits])

    # Normalize logits to probabilities and find top prob
    probs = np.exp(logit_values) / np.sum(np.exp(logit_values))    
    prediction = np.argmax(probs)

    if prediction == labels[i]: n_correct += 1

print(f"Accuracy: {(n_correct/n_test * 100):.2f}%")
    

Accuracy: 28.00%
